# Contract-Net Marketplace — Auction Experiment

Iterative prototyping with **qwen3.5:4b-mlx** via LangChain + Ollama.

**Flow:** Announcement → Bidding → Evaluation → Award → Execution → Feedback

In [1]:
import json, re, os
from typing import Any
from dataclasses import dataclass, field
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage

---
## 1. LLM Setup & clean_json()

In [2]:
llm = ChatOllama(model="qwen3.5:4b-mlx", temperature=0.0, format="json", reasoning=False)
print(f"Model: {llm.model}")

Model: qwen3.5:4b-mlx


In [3]:
def clean_json(raw: str) -> dict:
    """Extract and parse the first JSON object from raw LLM output."""
    raw = raw.strip()
    raw = re.sub(r'^```(?:json)?\s*', '', raw)
    raw = re.sub(r'\s*```$', '', raw)
    # Try direct parse
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        pass
    # Find ALL {...} blocks and try each
    for match in re.finditer(r'\{.*?\}', raw, re.DOTALL):
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            continue
    # Try greedy match as last resort
    match = re.search(r'\{.*\}', raw, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass
    raise ValueError(f"Could not extract JSON from: {raw[:200]!r}")

tests = [
    '{"a": 1}',
    '```json\n{"a": 1}\n```',
    '```\n{"a": 1}\n```',
    'Here is: {\"a\": 1}\n',
    '{"a": 1}\n{"b": 2}',
]
for t in tests:
    try:
        print(f"OK  {clean_json(t)}")
    except ValueError as e:
        print(f"FAIL {e}")

OK  {'a': 1}
OK  {'a': 1}
OK  {'a': 1}
OK  {'a': 1}
OK  {'a': 1}


In [4]:
# Quick connectivity test
resp = llm.invoke([
    SystemMessage(content="Always respond in JSON."),
    HumanMessage(content='Say hello. Respond: {"greeting": "..."}'),
])
data = clean_json(resp.content)
print(f"LLM: {data}")

LLM: {'greeting': 'Hello!'}


---
## 2. Improved Prompt — Stricter Domain Filtering

Issue from first run: agents claim they can handle tasks outside their domain.
Fix: make the instructions rigid — explicit `can_handle: false` requirement
when the task type is not in the domain list.

In [5]:
EVAL_SYSTEM = """You are {agent_id}, an AI agent specialized ONLY in: {domains}.

RULE: You must respond with can_handle=false if the task type is NOT in your domain list.
Do NOT assume you can do things outside your specialization.

Respond with this JSON:
{{
  "can_handle": true/false,
  "confidence": 0.0-1.0,
  "cost_dollars": 0.0,
  "eta_minutes": 0.0,
  "reasoning": "one sentence"
}}

- can_handle: MUST be false if task type not in your domains
- confidence: your skill level for this task
- cost_dollars: minimum $1 base + $5 per complexity point (max $100)
- eta_minutes: your estimated delivery time"""

EVAL_HUMAN = "Task type: {task_type}\nDescription: {task_desc}"

scenarios = [
    ("coder", ["code", "debugging"], "task-1", "Write a Python sort function", "code", True),
    ("poet",  ["writing", "poetry"],  "task-1", "Write a Python sort function", "code", False),
    ("coder", ["code", "debugging"], "task-2", "Train sentiment classifier", "ml", False),
    ("ml_eng",["ml", "data_science"],"task-2", "Train sentiment classifier", "ml", True),
    ("writer",["writing", "summarization"], "task-3", "Summarize a news article", "summarization", True),
    ("poet",  ["writing", "poetry"],  "task-3", "Summarize a news article", "summarization", False),
]

results = []
for agent_id, domains, task_id, task_desc, task_type, expected in scenarios:
    resp = llm.invoke([
        SystemMessage(content=EVAL_SYSTEM.format(agent_id=agent_id, domains=", ".join(domains))),
        HumanMessage(content=EVAL_HUMAN.format(task_type=task_type, task_desc=task_desc)),
    ])
    try:
        data = clean_json(resp.content)
        passed = data.get("can_handle") == expected
    except ValueError:
        data = {}
        passed = False
    results.append((agent_id, task_id, expected, data.get("can_handle"), passed, data))
    print(f"{'PASS' if passed else 'FAIL'}: {agent_id:8s} on {task_id} → can_handle={data.get('can_handle')} (expected {expected})")

total, passed = len(results), sum(1 for r in results if r[4])
print(f"\n=== Domain filter: {passed}/{total} passed ===")
print()
for r in results:
    d = r[5]
    print(f"  {r[0]:8s} | conf={str(d.get('confidence','?')):6s} | cost=${str(d.get('cost_dollars','?')):6s} | eta={str(d.get('eta_minutes','?')):6s}min | {d.get('reasoning','')[:80]}")

PASS: coder    on task-1 → can_handle=True (expected True)
PASS: poet     on task-1 → can_handle=False (expected False)
FAIL: coder    on task-2 → can_handle=True (expected False)
PASS: ml_eng   on task-2 → can_handle=True (expected True)
PASS: writer   on task-3 → can_handle=True (expected True)
PASS: poet     on task-3 → can_handle=False (expected False)

=== Domain filter: 5/6 passed ===

  coder    | conf=1.0    | cost=$5.0    | eta=5.0   min | Writing a Python sort function is directly within my domain of code generation a
  poet     | conf=0.0    | cost=$0.0    | eta=0.0   min | I am specialized only in writing and poetry, not in programming or code generati
  coder    | conf=0.95   | cost=$10.0   | eta=30.0  min | Training a sentiment classifier is a core machine learning task that falls direc
  ml_eng   | conf=0.95   | cost=$10.0   | eta=30.0  min | Training a sentiment classifier is a core data science and machine learning task
  writer   | conf=0.95   | cost=$6.0    | eta=15.

---
## 3. Execution with Inline Sample Data

Issue from first run: agents failed because no real data was provided.
Fix: include sample data inline in the task description.

In [6]:
EXEC_SYSTEM = """You are {agent_id}, specialized in: {domains}.
You have been awarded a task. Complete it and return the result.

Respond with this JSON:
{{
  "success": true/false,
  "output": "your result (be thorough)",
  "error": null or "error message"
}}"""

EXEC_HUMAN = "Task: {task_desc}"

exec_tests = [
    ("coder", ["code"],
     'Write a Python function add(a,b) that returns a+b. Include the code.'),
    ("writer", ["writing"],
     'Write a two-line poem about machine learning.'),
    ("ml_eng", ["ml"],
     'Analyze this data and predict next quarter sales:\n'
     'Q1: 100, Q2: 150, Q3: 200, Q4: 250. Provide your forecast and reasoning.'),
]

for agent_id, domains, task_desc in exec_tests:
    resp = llm.invoke([
        SystemMessage(content=EXEC_SYSTEM.format(agent_id=agent_id, domains=", ".join(domains))),
        HumanMessage(content=EXEC_HUMAN.format(task_desc=task_desc)),
    ])
    try:
        data = clean_json(resp.content)
        print(f"\n{agent_id}: success={data.get('success')}")
        if data.get('output'):
            print(f"  Output: {str(data['output'])[:250]}")
        if data.get('error'):
            print(f"  Error: {data['error']}")
    except ValueError:
        print(f"\n{agent_id}: PARSE FAILED: {resp.content[:150]!r}")


coder: success=True
  Output: def add(a, b):
    return a + b

# Example usage:
# result = add(3, 5)
# print(result)  # Output: 8

writer: success=True
  Output: From data's raw and chaotic stream,
It learns the patterns hidden in dreams.

ml_eng: success=True
  Output: ### Sales Forecast Analysis

**Data Provided:**
- Q1: 100 units
- Q2: 150 units
- Q3: 200 units
- Q4: 250 units

**Forecast for Next Quarter (Q5):**
**Predicted Sales: 300 units**

**Reasoning:**
1. **Trend Identification**: The data exhibits a consi


---
## 4. Utility Function & Reputation

In [7]:
@dataclass
class UtilityWeights:
    alpha: float = 1.0
    beta: float = 1.0
    gamma: float = 1.0

def utility_score(confidence: float, cost: float, eta_minutes: float, weights: UtilityWeights | None = None) -> float:
    w = weights or UtilityWeights()
    return w.alpha * confidence - w.beta * cost - w.gamma * (eta_minutes / 60.0)

bids = [("fast_cheap", 0.85, 15.0, 30.0), ("slow_expert", 0.99, 80.0, 120.0), ("budget", 0.70, 5.0, 600.0)]
for name, conf, cost, eta in bids:
    u = utility_score(conf, cost, eta)
    print(f"{name:12s} conf={conf:.2f} cost=${cost:.0f} eta={eta:.0f}min → utility={u:.2f}")
print()
# Time-sensitive weights
fast = UtilityWeights(alpha=1.0, beta=0.5, gamma=5.0)
for name, conf, cost, eta in bids:
    u = utility_score(conf, cost, eta, fast)
    print(f"{name:12s} (time-sensitive) → utility={u:.2f}")

fast_cheap   conf=0.85 cost=$15 eta=30min → utility=-14.65
slow_expert  conf=0.99 cost=$80 eta=120min → utility=-81.01
budget       conf=0.70 cost=$5 eta=600min → utility=-14.30

fast_cheap   (time-sensitive) → utility=-9.15
slow_expert  (time-sensitive) → utility=-49.01
budget       (time-sensitive) → utility=-51.80


In [8]:
class ReputationRegistry:
    """Tracks agent success/failure and adjusts confidence."""
    def __init__(self):
        self.scores: dict[str, dict[str, int]] = {}
    def record(self, agent_id: str, success: bool) -> None:
        s = self.scores.setdefault(agent_id, {"successes": 0, "failures": 0})
        s["successes" if success else "failures"] += 1
    def penalty(self, agent_id: str) -> float:
        s = self.scores.get(agent_id, {"successes": 0, "failures": 0})
        total = s["successes"] + s["failures"]
        return 0.0 if total == 0 else min(1.0, (s["failures"] / total) * 2.0)
    def adjust(self, agent_id: str, confidence: float) -> float:
        return confidence * (1.0 - self.penalty(agent_id))

reg = ReputationRegistry()
reg.record("coder", True)
reg.record("coder", True)
reg.record("coder", False)
print(f"coder: penalty={reg.penalty('coder'):.2f} adj_conf(0.9)={reg.adjust('coder', 0.9):.2f}")
reg.record("coder", False)
print(f"coder (4th): penalty={reg.penalty('coder'):.2f} adj_conf(0.9)={reg.adjust('coder', 0.9):.2f}")

coder: penalty=0.67 adj_conf(0.9)=0.30
coder (4th): penalty=1.00 adj_conf(0.9)=0.00


---
## 5. Full Auction Simulation

In [9]:
@dataclass
class Task:
    task_id: str
    description: str
    task_type: str
    constraints: dict = field(default_factory=lambda: {"max_cost": 100.0, "max_eta_hours": 4})

@dataclass
class Bid:
    agent_id: str
    confidence: float
    cost: float
    eta_minutes: float
    reasoning: str

@dataclass
class ContractResult:
    agent_id: str
    task_id: str
    success: bool
    output: str
    error: str | None

class NoBidsException(Exception):
    pass

In [10]:
class OllamaBidderAgent:
    def __init__(self, agent_id: str, domains: list[str], llm: ChatOllama):
        self.agent_id = agent_id
        self.domains = domains
        self.llm = llm
    
    def evaluate(self, task: Task) -> Bid | None:
        system = EVAL_SYSTEM.format(agent_id=self.agent_id, domains=", ".join(self.domains))
        human = EVAL_HUMAN.format(task_type=task.task_type, task_desc=task.description)
        resp = self.llm.invoke([SystemMessage(content=system), HumanMessage(content=human)])
        try:
            data = clean_json(resp.content)
        except ValueError:
            print(f"  [WARN] {self.agent_id}: JSON parse failed")
            print(f"  Raw: {resp.content[:150]!r}")
            return None
        if not data.get("can_handle", False):
            return None
        return Bid(
            agent_id=self.agent_id,
            confidence=float(data.get("confidence", 0.5)),
            cost=float(data.get("cost_dollars", 0)),
            eta_minutes=float(data.get("eta_minutes", 0)),
            reasoning=str(data.get("reasoning", "")),
        )
    
    def execute(self, task: Task) -> ContractResult:
        system = EXEC_SYSTEM.format(agent_id=self.agent_id, domains=", ".join(self.domains))
        human = EXEC_HUMAN.format(task_desc=task.description)
        resp = self.llm.invoke([SystemMessage(content=system), HumanMessage(content=human)])
        try:
            data = clean_json(resp.content)
        except ValueError:
            return ContractResult(self.agent_id, task.task_id, False, "", "JSON parse failed")
        return ContractResult(
            agent_id=self.agent_id,
            task_id=task.task_id,
            success=bool(data.get("success", False)),
            output=str(data.get("output", "")),
            error=str(data.get("error")) if data.get("error") else None,
        )

In [11]:
class Solicitor:
    def __init__(self, agents: list[OllamaBidderAgent], reputation: ReputationRegistry, weights: UtilityWeights | None = None):
        self.agents = agents
        self.reputation = reputation
        self.weights = weights or UtilityWeights()
    
    def run_auction(self, task: Task) -> tuple[OllamaBidderAgent, Bid, ContractResult]:
        print(f"\n{'='*60}")
        print(f"  TASK: {task.description}")
        print(f"  Type: {task.task_type}")
        print(f"{'='*60}")
        
        bids: list[tuple[OllamaBidderAgent, Bid]] = []
        for agent in self.agents:
            print(f"  \n  {agent.agent_id}:", end="")
            bid = agent.evaluate(task)
            if bid is None:
                print(f" CANNOT_HANDLE")
                continue
            adj_conf = self.reputation.adjust(agent.agent_id, bid.confidence)
            print(f" bids ${bid.cost:.0f} | {bid.confidence:.0%} conf (adj: {adj_conf:.0%}) | {bid.eta_minutes:.0f}min")
            if bid.reasoning:
                print(f"    → {bid.reasoning[:100]}")
            bids.append((agent, bid))
        
        if not bids:
            raise NoBidsException(f"No agent can handle: {task.description}")
        
        def score(item):
            a, b = item
            return utility_score(self.reputation.adjust(a.agent_id, b.confidence), b.cost, b.eta_minutes, self.weights)
        
        bids.sort(key=score, reverse=True)
        winner_agent, winner_bid = bids[0]
        
        print(f"\n  ★ WINNER: {winner_agent.agent_id} (utility={score(bids[0]):.2f})")
        
        result = winner_agent.execute(task)
        self.reputation.record(winner_agent.agent_id, result.success)
        
        emoji = "✅" if result.success else "❌"
        print(f"  Result: {emoji} {'SUCCESS' if result.success else 'FAILURE'}")
        if result.output:
            trunc = result.output[:400]
            print(f"  Output: {trunc}")
        if result.error:
            print(f"  Error: {result.error}")
        
        with open("reputation.json", "w") as f:
            json.dump(self.reputation.scores, f, indent=2)
        
        return winner_agent, winner_bid, result

In [12]:
agents = [
    OllamaBidderAgent("coder",  ["code", "debugging"], llm),
    OllamaBidderAgent("writer", ["writing", "poetry", "summarization"], llm),
    OllamaBidderAgent("ml_eng", ["ml", "data_science"], llm),
]
reputation = ReputationRegistry()
if os.path.exists("reputation.json"):
    with open("reputation.json") as f:
        reputation.scores = json.load(f)
        print(f"Loaded reputation: {reputation.scores}")

solicitor = Solicitor(agents, reputation)

tasks = [
    Task("t1", "Write a Python function to sort integers using quicksort", "code"),
    Task("t2", "Write a haiku about gradient descent", "poetry"),
    Task("t3", "Predict Q1 sales given: Q1=100, Q2=150, Q3=200, Q4=250", "data_science"),
    Task("t4", "Fix bug: this JS function has a typo. function greet(name {{ return 'Hello ' + name }}", "debugging"),
]

for task in tasks:
    try:
        solicitor.run_auction(task)
    except NoBidsException as e:
        print(f"\n  ⚠ NO BIDS: {e}")

Loaded reputation: {'coder': {'successes': 2, 'failures': 0}, 'writer': {'successes': 1, 'failures': 0}}

  TASK: Write a Python function to sort integers using quicksort
  Type: code
  
  coder: bids $5 | 100% conf (adj: 100%) | 5min
    → I can write a Python quicksort function as it falls directly within my domain of code generation and
  
  writer: CANNOT_HANDLE
  
  ml_eng: bids $5 | 100% conf (adj: 100%) | 5min
    → Quicksort is a fundamental algorithm in data science and computer science that fits perfectly within

  ★ WINNER: coder (utility=-4.08)
  Result: ✅ SUCCESS
  Output: def quicksort(arr):
    if len(arr) <= 1:
        return arr
    
    pivot = arr[len(arr) // 2]
    left = [x for x in arr if x < pivot]
    middle = [x for x in arr if x == pivot]
    right = [x for x in arr if x > pivot]
    
    return quicksort(left) + middle + quicksort(right)

# Example usage:
# print(quicksort([3, 6, 2, 1]))
# Output: [1, 2, 3, 6]

  TASK: Write a haiku about gradient descent
  T

### 5.1 Edge case test

In [13]:
try:
    solicitor.run_auction(Task("t5", "Perform open-heart surgery", "medical"))
except NoBidsException as e:
    print(f"\n  ✓ NoBidsException raised: {e}")

print(f"\nFinal reputation: {reputation.scores}")
if os.path.exists("reputation.json"):
    with open("reputation.json") as f:
        print(f"Persisted: {json.load(f)}")


  TASK: Perform open-heart surgery
  Type: medical
  
  coder: CANNOT_HANDLE
  
  writer: CANNOT_HANDLE
  
  ml_eng: CANNOT_HANDLE

  ✓ NoBidsException raised: No agent can handle: Perform open-heart surgery

Final reputation: {'coder': {'successes': 4, 'failures': 0}, 'writer': {'successes': 2, 'failures': 0}, 'ml_eng': {'successes': 1, 'failures': 0}}
Persisted: {'coder': {'successes': 4, 'failures': 0}, 'writer': {'successes': 2, 'failures': 0}, 'ml_eng': {'successes': 1, 'failures': 0}}


---
## 6. Summary

After the experiments above:
- Domain filtering accuracy (prompt v1 vs v2)
- JSON reliability with `format="json"`
- Whether cost/eta estimates are useful
- Execution quality on tasks with inline data
- Edge cases: no-bid handling, reputation persistence